In [18]:
import json
import unicodedata
from pathlib import Path

import pandas as pd
import numpy as np
from scipy.stats import spearmanr

In [9]:
TARGET_WORDS = [
    "überspannen",
    "Manschette",
    "Fuß",
    "Rezeption",
    "abgebrüht",
    "Dynamik",
    "Engpaß",
    "abbauen",
    "Mißklang",
    "Abgesang",
    "Knotenpunkt",
    "Spielball",
    "zersetzen",
    "Armenhaus",
    "Ohrwurm",
    "Eintagsfliege",
    "Seminar",
    "Sensation",
    "Titel",
    "Schmiere",
    "ausspannen",
    "packen",
    "artikulieren",
    "abdecken",
]

In [10]:
annotations_schemas = {
    "en-en": "schemas_for_german/en-en/german",
    "ru-ru": "schemas_for_german/ru-ru/german",
    "rusemshift-finetune": "schemas_for_german/rusemshift/finetune/german",
    "rusemshift-train": "schemas_for_german/rusemshift/train/german",
}

In [11]:
def decode_hash_unicode(name: str) -> str:
    return name.replace("u#U0308", "ü").replace("#U00df", "ß")

In [12]:
def load_scores_for_word(
    schema_base_path: str,
    word: str,
    pairs_base_path: str,
    context_to_grouping: dict,
) -> pd.DataFrame:

    base = Path(schema_base_path)
    word_nfc = unicodedata.normalize("NFC", word)

    word_dir = None
    for subdir in base.iterdir():
        if subdir.is_dir():
            decoded = unicodedata.normalize("NFC", decode_hash_unicode(subdir.name))
            if decoded == word_nfc:
                word_dir = subdir
                break

    if word_dir is None:
        print(f"  Directory not found for: {word}")
        return None

    score_files = list(word_dir.glob("*.scores"))
    if not score_files:
        print(f"  No scores file in: {word_dir}")
        return None

    scores_data = pd.read_json(score_files[0])
    scores_data["score"] = scores_data["score"].apply(
        lambda x: np.mean([float(v) for v in x])
    )

    word_encoded = word_dir.name
    pair_file = Path(pairs_base_path) / word_encoded / f"dev.{word_encoded}.data"
    if not pair_file.exists():
        print(f"  Pair file not found for: {word}")
        return None

    pairs = pd.read_json(pair_file)
    merged = pd.merge(pairs, scores_data, on="id")
    print(f"  Merged rows: {len(merged)}")
    
    sample_sentence = merged["sentence1"].iloc[0]
    print(f"  Sample sentence1: {repr(sample_sentence[:80])}")
    print(f"  Found in context_to_grouping: {sample_sentence in context_to_grouping}")
    
    mapped1 = merged["sentence1"].map(context_to_grouping).notna().sum()
    mapped2 = merged["sentence2"].map(context_to_grouping).notna().sum()
    print(f"  Mapped sentence1: {mapped1}/{len(merged)}")
    print(f"  Mapped sentence2: {mapped2}/{len(merged)}")
    
    merged["grouping1"] = merged["sentence1"].map(context_to_grouping)
    merged["grouping2"] = merged["sentence2"].map(context_to_grouping)
    
    cross = merged[merged["grouping1"] != merged["grouping2"]]
    print(f"  {word}: total={len(merged)} cross-perid={len(cross)}")
    
    if len(cross) == 0:
        print(f"  No cross-period pairs for: {word}")
        return None
    
    return cross[["score"]]
    
    

In [13]:
def compute_apd(scores_df: pd.DataFrame) -> float:
    if scores_df is None or len(scores_df) == 0:
        return None

    return float((1 - scores_df["score"]).mean())

In [14]:
old_senses = pd.read_csv(
    "summer-wsi/datasets_unlabeled/se20lscd_v2/de/unlabeled-old.tsv",
    sep="\t",
)
new_senses = pd.read_csv(
    "summer-wsi/datasets_unlabeled/se20lscd_v2/de/unlabeled-new.tsv",
    sep="\t",
)

context_to_grouping = {
    **dict(zip(old_senses["context"], [1] * len(old_senses))),
    **dict(zip(new_senses["context"], [2] * len(new_senses))),
}

pairs_base_path = "Serge/german"
print(f"Old: {len(old_senses)}, New: {len(new_senses)}, Total: {len(context_to_grouping)}")


Old: 4125, New: 5000, Total: 8978


In [15]:
results = {}

for schema_name, schema_path in annotations_schemas.items():
    print(f"\n=== {schema_name} ===")
    results[schema_name] = {}

    for word in TARGET_WORDS:
        scores_df = load_scores_for_word(
            schema_path,
            word,
            pairs_base_path,
            context_to_grouping,
        )
        apd = compute_apd(scores_df)
        results[schema_name][word] = apd
        if apd is not None:
            print(f"  {word}: APD = {apd:.4f}")
        else:
            print(f"  {word}: no data")


=== en-en ===
  Merged rows: 19808
  Sample sentence1: 'Wenn wir zu dieser Absicht den halben Kreis in vier gleiche Theile theilen, und '
  Found in context_to_grouping: True
  Mapped sentence1: 19808/19808
  Mapped sentence2: 19808/19808
  überspannen: total=19808 cross-perid=9970
  überspannen: APD = 0.5124
  Merged rows: 15176
  Sample sentence1: 'Und grade ein schlagendes Beispiel dieser Inquisition, in Frack und Manschetten,'
  Found in context_to_grouping: True
  Mapped sentence1: 15176/15176
  Mapped sentence2: 15176/15176
  Manschette: total=15176 cross-perid=7505
  Manschette: APD = 0.4909
  Merged rows: 19900
  Sample sentence1: 'Der goldene Schuh an deinem Fuß, Er ist’s, der dich erlösen muß.'
  Found in context_to_grouping: True
  Mapped sentence1: 19900/19900
  Mapped sentence2: 19900/19900
  Fuß: total=19900 cross-perid=10000
  Fuß: APD = 0.5233
  Merged rows: 19462
  Sample sentence1: 'Das Staatsrecht des deutschen Reiches und seiner Territorien war durch die Rezep'
  F

In [16]:
def load_gold_data(path: str) -> dict:
    df = pd.read_csv(path, sep="\t")
    try:
        return dict(zip(df["lemma"], df["change_graded"]))
    except KeyError:
        return dict(zip(df["word"], df["change_graded"]))


gold_data = load_gold_data("gold-data-de.csv")

print("Spearman correlations: ")
for schema_name in annotations_schemas:
    pairs = [
        (results[schema_name][w], gold_data[w])
        for w in TARGET_WORDS
        if results[schema_name].get(w) is not None and w in gold_data
    ]
    if len(pairs) < 2:
        print(f"  {schema_name}: not enough data")
        continue

    apd_values, gold_values = zip(*pairs)
    spearman, pvalue = spearmanr(gold_values, apd_values)
    print(f"  {schema_name}: Spearman = {spearman:.4f} (p = {pvalue:.4f})")

Spearman correlations: 
  en-en: Spearman = 0.5400 (p = 0.0065)
  ru-ru: Spearman = 0.3635 (p = 0.0808)
  rusemshift-finetune: Spearman = 0.5843 (p = 0.0027)
  rusemshift-train: Spearman = 0.5052 (p = 0.0118)


In [ ]:
rows = []
for schema_name in annotations_schemas:
    for word in TARGET_WORDS:
        rows.append(
            {
                "schema": schema_name,
                "word": word,
                "apd": results[schema_name].get(word),
            }
        )

summary_df = pd.DataFrame(rows)
summary_df.pivot(index="word", columns="schema", values="apd").round(4)

schema,en-en,ru-ru,rusemshift-finetune,rusemshift-train
word,,,,
Abgesang,0.4995,0.4990,0.5034,0.5052
Armenhaus,0.5106,0.4994,0.5138,0.5031
Dynamik,0.5264,0.5058,0.5141,0.5131
Eintagsfliege,0.5012,0.4952,0.5059,0.5016
Engpaß,0.5339,0.5104,0.5231,0.5205
Fuß,0.5233,0.5108,0.5178,0.5153
Knotenpunkt,0.5183,0.5050,0.5173,0.5089
Manschette,0.4909,0.4909,0.4900,0.4954
Mißklang,0.4936,0.4993,0.5002,0.5043


In [ ]:
folds = json.loads(Path("./folds_german_dataset.json").read_text())

cv_results = {}

for schema_name, schema_path in annotations_schemas.items():
    print(f"\n=== {schema_name=} ===")
    cv_results[schema_name] = []
    
    for fold_name, fold_data in folds.items():
        test_words = [w for w in fold_data["test"] if w in TARGET_WORDS]
        
        apd_per_word = {}
        for word in test_words:
            scores_df = load_scores_for_word(schema_path, word, pairs_base_path, context_to_grouping)
            apd = compute_apd(scores_df)
            if apd is not None:
                apd_per_word[word] = apd
                
        pairs = [
            (apd_per_word[w], gold_data[w])
            for w in test_words
            if w in apd_per_word and w in gold_data
        ]
        
        if len(pairs) < 2:
            print(f"  {fold_name}: not enough data")
            continue
        
        apd_values, gold_values = zip(*pairs)
        spearman, _ = spearmanr(gold_values, apd_values)
        cv_results[schema_name].append(spearman)
        print(f"  {fold_name}: Spearman = {spearman:.4f} ({len(pairs)} words)")
        
        

{'fold_1': {'train': ['überspannen',
   'Manschette',
   'Fuß',
   'Rezeption',
   'abgebrüht',
   'Dynamik',
   'Engpaß',
   'abbauen',
   'Abgesang',
   'Knotenpunkt',
   'Spielball',
   'zersetzen',
   'Armenhaus',
   'Eintagsfliege',
   'Titel',
   'Schmiere',
   'packen',
   'artikulieren',
   'abdecken'],
  'test': ['Mißklang', 'Ohrwurm', 'Seminar', 'Sensation', 'ausspannen']},
 'fold_2': {'train': ['Manschette',
   'Rezeption',
   'abgebrüht',
   'Dynamik',
   'abbauen',
   'Mißklang',
   'Abgesang',
   'Knotenpunkt',
   'Spielball',
   'zersetzen',
   'Ohrwurm',
   'Seminar',
   'Sensation',
   'Titel',
   'Schmiere',
   'ausspannen',
   'packen',
   'artikulieren',
   'abdecken'],
  'test': ['überspannen', 'Fuß', 'Engpaß', 'Armenhaus', 'Eintagsfliege']},
 'fold_3': {'train': ['überspannen',
   'Fuß',
   'Rezeption',
   'Dynamik',
   'Engpaß',
   'abbauen',
   'Mißklang',
   'Abgesang',
   'Knotenpunkt',
   'Spielball',
   'Armenhaus',
   'Ohrwurm',
   'Eintagsfliege',
   'Semi

In [ ]:
print(f"\nCV Summary (avg Spearman across folds):")
for schema_name, scores in cv_results.items():
    if scores:
        print(f"  {schema_name}: {np.mean(scores):.4f} += {np.std(scores):.4f}")
    else:
        print(f"  {schema_name}: no data")